# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadOmerSiddiqui/myInternship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [5]:
import os, sys, subprocess
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    REPO = "myInternship"
    if not os.path.isdir(REPO):
        subprocess.run(
            ["git", "clone", "--depth", "1",
             "https://github.com/MuhammadOmerSiddiqui/myInternship.git", REPO],
            check=True
        )
    os.chdir(REPO)

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining"] = (df["trend_direction"] == "down").astype(int)
print("Loaded", len(df), "pages | declining rate:", round(df["is_declining"].mean(), 3))

Loaded 30000 pages | declining rate: 0.542


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

I read FlyRank’s public study *The State of AI-Driven SEO* (March 2026, in `docs/flyrank-seo-research-march-2026.pdf`). Below are two findings and the methodology questions I would ask — respectfully, the way I want my own work reviewed.


**Finding A — Growing vs declining pages differ on depth, age, and visibility**  
The paper reports that pages with rising impressions tend to be longer, younger, and slightly better positioned than pages with falling impressions (Finding #1).

**Methodology question I would ask:**  
Where exactly does the “growing / falling” label come from — is it last-30d vs prev-30d impressions only, and were seasonality or site-level trends controlled? Does the comparison support a causal “refresh younger pages” claim, or only an observed association?


**Finding B — Weighted CTR falls sharply away from top positions**  
Portfolio weighted CTR is much higher in top-3 / page-1 tiers than in deeper positions (Finding #3). The paper treats this as a reason to refine page-one snippets before rebuilding weak pages.

**Methodology question I would ask:**  
Is the CTR comparison conditioned only on position tier, or also on minimum impression volume? Without a volume floor, low-impression deep pages can distort tier averages. Does the validation design (portfolio aggregates) support the action recommendation, or should it stay as a descriptive pattern until a before/after test is run?


Tone: these are constructive questions, not a grade. The paper already frames ML sections as exploratory and pairs scores with raw search metrics — that honesty is the standard I apply to my own model next.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

**Before:** random row split (stratified) — easier, but clients can leak across train/test.  
**After:** client-holdout — no page from a test client appears in training.

Same features, same label, same model (Random Forest), same metric (Precision@50).  
The gap between the two numbers shows how much “skill” was client memorization.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
RANDOM_STATE = 42

# ----- features (honest; no trend/label) -----
numeric_cols = [c for c in [
    "impressions_90d","clicks_90d","sessions_90d","content_age_days",
    "days_since_last_update","avg_position","ctr","engagement_rate","scroll_rate",
    "word_count","char_count","days_with_impressions","days_with_sessions",
    "ai_sessions_90d","ai_traffic_pct"
] if c in df.columns]
cat_cols = [c for c in [
    "content_type","main_intent","competition_level",
    "age_tier","freshness_tier","impression_tier","position_tier"
] if c in df.columns]

X_num = df[numeric_cols].apply(pd.to_numeric, errors="coerce").replace([np.inf,-np.inf],np.nan).fillna(0)
X_cat = pd.get_dummies(df[cat_cols].fillna("unknown").astype(str), dtype=float)
X = pd.concat([X_num.reset_index(drop=True), X_cat.reset_index(drop=True)], axis=1)
y = df["is_declining"].astype(int)

def precision_at_k(y_true, scores, k=50):
    order = np.argsort(-np.asarray(scores))
    return float(np.asarray(y_true)[order[:k]].mean())

def run_rf(X_tr, y_tr, X_te, y_te):
    model = RandomForestClassifier(
        class_weight="balanced_subsample", n_estimators=200, max_depth=10,
        min_samples_leaf=25, n_jobs=-1, random_state=RANDOM_STATE
    )
    model.fit(X_tr, y_tr)
    proba = model.predict_proba(X_te)[:, 1]
    return {
        "precision@50": round(precision_at_k(y_te, proba, 50), 3),
        "roc_auc": round(roc_auc_score(y_te, proba), 3),
        "avg_precision": round(average_precision_score(y_te, proba), 3),
        "test_base_rate": round(float(y_te.mean()), 3),
        "test_rows": len(y_te),
    }, model, proba

# ===== BEFORE: random stratified split =====
idx = np.arange(len(df))
tr_rand, te_rand = train_test_split(
    idx, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
before, _, _ = run_rf(X.iloc[tr_rand], y.iloc[tr_rand], X.iloc[te_rand], y.iloc[te_rand])

# ===== AFTER: client-holdout =====
rng = np.random.default_rng(RANDOM_STATE)
clients = df["client_id"].fillna("unknown").astype(str)
uniq = clients.drop_duplicates().to_numpy()
shuf = rng.permutation(uniq)
n_test = max(1, int(round(len(shuf) * 0.20)))
test_clients = set(shuf[:n_test])
test_mask = clients.isin(test_clients).to_numpy()
tr_cli = np.where(~test_mask)[0]
te_cli = np.where(test_mask)[0]
after, rf_model, te_proba = run_rf(X.iloc[tr_cli], y.iloc[tr_cli], X.iloc[te_cli], y.iloc[te_cli])

compare = pd.DataFrame([
    {"split": "random_row (BEFORE)", **before},
    {"split": "client_holdout (AFTER)", **after},
])
print("Base rate overall:", round(y.mean(), 3))
display(compare)
print("""
Reading the gap:
- If random >> client-holdout, the model was partly memorizing clients.
- Client-holdout is the honest number for 'does this help on new clients?'
- We keep client-holdout as the validation design for the paper.
""")

Base rate overall: 0.542


,split,precision@50,roc_auc,avg_precision,test_base_rate,test_rows
0,random_row (BEFORE),0.90,0.757,0.768,0.542,6000
1,client_holdout (AFTER),0.82,0.749,0.620,0.391,2325



Reading the gap:
- If random >> client-holdout, the model was partly memorizing clients.
- Client-holdout is the honest number for 'does this help on new clients?'
- We keep client-holdout as the validation design for the paper.



## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

**Checks run on the final feature set**

| Risk | Check | Result |
|------|--------|--------|
| Label-derived features | Is `trend_direction` / `trend_pct` / `is_declining` in X? | No |
| Future window | All metrics are trailing-90d snapshot fields available at export | OK for this proxy label |
| Product flags | health_score / priority_score / action_type in data? | Not present |
| Deliberate leak test | Add `is_declining` as a feature → score jumps → remove it | Shown below |

Conclusion: no label leakage in the production feature matrix. The proxy label itself is current-window (limitation, not leakage).

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("=== Leakage audit ===")
forbidden = {"trend_direction", "trend_pct", "is_declining", "is_declining_label"}
overlap = forbidden.intersection(set(X.columns))
print("Forbidden columns in feature matrix:", overlap if overlap else "NONE (good)")

# Deliberate leak test
X_leak = X.copy()
X_leak["LEAK_is_declining"] = y.values
# use client-holdout indices
m_leak, _, _ = run_rf(X_leak.iloc[tr_cli], y.iloc[tr_cli], X_leak.iloc[te_cli], y.iloc[te_cli])
m_honest = after  # from previous cell

print("\nHonest RF Precision@50 :", m_honest["precision@50"])
print("With deliberate leak   :", m_leak["precision@50"], "← jumps toward perfect")
print("→ Leak column removed. We keep only the honest feature set.")

# Top importances (sanity)
imp = pd.Series(rf_model.feature_importances_, index=X.columns).sort_values(ascending=False)
print("\nTop 8 features (should look like volume/age/position — not a single magic column):")
display(imp.head(8).to_frame("importance"))

=== Leakage audit ===
Forbidden columns in feature matrix: NONE (good)

Honest RF Precision@50 : 0.82
With deliberate leak   : 1.0 ← jumps toward perfect
→ Leak column removed. We keep only the honest feature set.

Top 8 features (should look like volume/age/position — not a single magic column):


,importance
days_with_impressions,0.152420
impressions_90d,0.126260
avg_position,0.119697
content_age_days,0.090008
char_count,0.046953
word_count,0.046721
age_tier_365+,0.039058
clicks_90d,0.038662


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Bold claim I might have written earlier (too strong):**  
“Our model predicts which pages will decline and tells editors exactly what to refresh so traffic recovers.”

**Rewritten safe claim (what the evidence supports):**  
“On a client-holdout split of the 30k-page starter dataset, a Random Forest ranking beat a transparent hand-written rule on Precision@50 for a current-window declining proxy. The output is a decision-support queue with reason codes for human review. It does not prove that a refresh causes recovery, and it does not predict Google’s algorithm.”

**Language rules I will keep in the paper**
- observed / measured / directional / decision-support  
- never “causes”, “will recover”, “Google’s algorithm”  
- always report base rate next to Precision@K

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Real failure examples under client-holdout (supports honest claims)
test_frame = df.iloc[te_cli].copy()
test_frame["model_score"] = te_proba
top50 = test_frame.nlargest(50, "model_score")

print("Top-50 of model | declining count:", int(top50["is_declining"].sum()), "/ 50")
print("False positives in top-50:", int((top50["is_declining"] == 0).sum()))

fp = top50[top50["is_declining"] == 0][
    ["content_id", "impressions_90d", "avg_position", "ctr",
     "content_age_days", "trend_direction", "model_score"]
].head(5)
print("\nExample false positives (high score, not declining):")
display(fp)

print("""
These errors are why claims stay decision-support only:
high-volume stable pages can still rank high. A human must apply context
(seasonality, consolidation) before acting.
""")

Top-50 of model | declining count: 41 / 50
False positives in top-50: 9

Example false positives (high score, not declining):


,content_id,impressions_90d,avg_position,ctr,content_age_days,trend_direction,model_score
4249,content_db1cd41b4b4f,1482,12.9,0.00,105,up,0.770469
23750,content_e55b8ab078b0,369,21.8,0.00,112,stable,0.744038
5966,content_f5013794ba57,881,15.7,0.00,175,new,0.739428
19812,content_ac140b295c0f,1012,26.0,0.00,175,stable,0.736463
23559,content_00603b0349b4,1076,25.6,0.09,125,up,0.734915



These errors are why claims stay decision-support only:
high-volume stable pages can still rank high. A human must apply context
(seasonality, consolidation) before acting.



## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.